## Structured Output

Models can be requested to provide their response in a format matching a given schema. This is useful for ensuring the output can be easily parsed and used in subsequent processing. Lanchain supports multiple schema types and methods for enforcing structured output.


### Pydantic

Pydantic models provide the richest feature set with field validation, descriptions, and nested structures.

In [2]:
import os
from langchain_openai import ChatOpenAI

model = ChatOpenAI(
    model = "nvidia/nemotron-3-super-120b-a12b:free",
    api_key = os.environ["OPENROUTER_API_KEY"],
    base_url = "https://openrouter.ai/api/v1"
)

- Here what we are doing is we are creating a structure for the LLM to follow when giving out an output, and it will be validated using pydantic

In [3]:
from pydantic import BaseModel,Field

## We have created fields with particular data type thus it will be validated by pydantic
class Movie(BaseModel):
    title:str = Field(description="This is the title of the movie")
    year:int = Field(description="This year the movie was released")
    director:str = Field(description="This is the director of the movie")
    rating:float = Field(description="The movie's ratings out of 10")


In [5]:
model_with_structured_output = model.with_structured_output(Movie)
model_with_structured_output

_ChatModelBinding(bound=ChatOpenAI(metadata={'lc_versions': {'langchain-core': '1.4.7', 'langchain': '1.3.9', 'langchain-openai': '1.3.2'}}, output_version=None, client=<openai.resources.chat.completions.completions.Completions object at 0x1156852b0>, async_client=<openai.resources.chat.completions.completions.AsyncCompletions object at 0x115685d30>, root_client=<openai.OpenAI object at 0x114ab0590>, root_async_client=<openai.AsyncOpenAI object at 0x115685a90>, model_name='nvidia/nemotron-3-super-120b-a12b:free', model_kwargs={}, openai_api_key=SecretStr('**********'), openai_api_base='https://openrouter.ai/api/v1', openai_proxy=None, stream_chunk_timeout=120.0), kwargs={'response_format': <class '__main__.Movie'>, 'ls_structured_output_format': {'kwargs': {'method': 'json_schema', 'strict': None}, 'schema': {'type': 'function', 'function': {'name': 'Movie', 'description': '', 'parameters': {'properties': {'title': {'description': 'This is the title of the movie', 'type': 'string'}, 'y

In [7]:
model.invoke("Provide details about the movie Dil wale dulhaniya le jayenge")

KeyboardInterrupt: 

In [ ]:
model_with_structured_output.invoke("Provide details about the movie Dil wale dulhaniya le jayenge")

Movie(title='Dilwale Dulhania Le Jayenge (1995)', year=1995, director='Aditya Chopra (directorial debut) ', rating=8.1)

### Message Output alongside parsed structure

In [8]:
from pydantic import BaseModel, Field

class Movie(BaseModel):
    """A movie with details"""
    title:str = Field(..., description="The title of the movie")
    year :int= Field(..., description="This is the year the movie was released")
    director : str = Field(..., description="The director of the movie")
    rating : float = Field(..., description="The movie's rating out of 10")

model_with_structure = model.with_structured_output(Movie, include_raw = True)

In [10]:
response = model_with_structure.invoke("Please provide details about the movie me before you")
response

{'raw': AIMessage(content='{"title":"Me Before You (2016)","year":2016,"director":"Thea Sharrock,","rating":3.5452129435810284}', additional_kwargs={'parsed': Movie(title='Me Before You (2016)', year=2016, director='Thea Sharrock,', rating=3.5452129435810282), 'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 621, 'prompt_tokens': 25, 'total_tokens': 646, 'completion_tokens_details': {'accepted_prediction_tokens': None, 'audio_tokens': 0, 'reasoning_tokens': 674, 'rejected_prediction_tokens': None, 'image_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0, 'cache_write_tokens': 0, 'video_tokens': 0}, 'cost': 0, 'is_byok': False, 'cost_details': {'upstream_inference_cost': 0, 'upstream_inference_prompt_cost': 0, 'upstream_inference_completions_cost': 0}}, 'model_provider': 'openai', 'model_name': 'nvidia/nemotron-3-super-120b-a12b-20230311:free', 'system_fingerprint': None, 'id': 'gen-1782858236-BJCZ8K4IYoEcW4p2mhLY', 'finish_reason': 's

- So what essentially is happening with pydantic is that we are able to create a datatype of out own and use it to define how the LLM dishes out responses

In [ ]:
from pydantic import BaseModel, Field

class Actor(BaseModel):
    name:str
    role:str

class MovieDetails(BaseModel):
    title:str
    year:int
    cast : list[Actor]
    genres: list[str]
    budget : float | None = Field(None, description="Budget in millions USD")

model_with_structures = model.with_structured_output(MovieDetails)

response = model_with_structures.invoke("Provide details about the movie Inception")

## TypeDict

TypeDict provides a simpler alternative using Python's built in typing, ideal when you don't need a runtime validation

In [13]:
from typing_extensions import TypedDict, Annotated


class MovieDict(TypedDict):
    """A movie with details"""
    title: Annotated[str,..., "The title of the movie"]
    year : Annotated[str,..., "The year the movie was released"]
    director : Annotated[str, ..., "The director of the movie"]
    rating : Annotated[float, ... ,"The movie's rating out of 10"]
    


In [16]:
new_model = model.with_structured_output(MovieDict, strict = True)

In [17]:
new_model.invoke("Please provide the details of the movie avengers")

{'title': 'Avengers (2012)',
 'year': '2012 ',
 'director': 'Joss Whedon ',
 'rating': 8.0}

### DataClasses
You can create is directly using the @dataclass decorator

In [25]:
from dataclasses import dataclass
from langchain.agents import create_agent

@dataclass
class ContactInfo:
    """Contact information for a person"""
    name : str
    email: str
    phone : str

agent = create_agent(
    model = model,
    response_format = ContactInfo
)

result = agent.invoke({
    "messages" : [{"role" : "user", "content" : "Extract contact info from : John Doe, john@example.com, (555) 123-4567"}]
})

result["structured_response"]

ContactInfo(name='John Doe', email='john@example.com', phone='(555) 123-4567')